## Imports

In [88]:
# general
import random
import numpy as np
import pandas as pd
from collections import defaultdict
import random

## Data handling


In [124]:
def tokenize(txtFile):
    stop_sentence_tokens = [".", "!", "?"]
    interrupt_sentence_tokens = [",", ";", ":"]

    words = ["<s>"]
    word = []
    for character in txtFile.read().lower():
        if character == "\n":
            if word:
                words.append("".join(word))
                words.append("</s>")
                words.append("<s>")            
                word = []
            continue    
        if character == " ":
            if word:
                words.append("".join(word))
                word = []
            continue
        if character in stop_sentence_tokens:
            if word:
                words.append("".join(word))
                words.append(character)
                words.append("</s>")
                words.append("<s>")
                word = []
            continue
        if character in interrupt_sentence_tokens:
            if word:
                words.append("".join(word))
                words.append(character)
                word = []
            continue
        word.append(character)
    return words

def vocabulary(words):
    return list(set(words))

def sentenceinizer(words):
    sentences = []
    sentence = []
    for word in words:
        sentence.append(word)
        if word == "</s>":
            sentences.append(sentence)
            sentence = []
    return sentences

def to_trigram_data(sentences):
    temp_sentences = [list(sentence) for sentence in sentences]
    for sentence in temp_sentences:
        sentence.insert(0, "<s>")
    return temp_sentences

In [129]:
txtFile = open("data/lotr.txt")
words = tokenize(txtFile)
vocab = vocabulary(words)
sentences = sentenceinizer(words)
trigram_sentences = to_trigram_data(sentences)

## BiGram

In [ ]:
def compute_bigram_frequencies(sentences):
    bigram_frequencies = {}

    for sentence in sentences:
        for w1, w2 in zip(sentence, sentence[1:]):
            
            if w1 not in bigram_frequencies:
                bigram_frequencies[w1] = {}
            
            bigram_frequencies[w1][w2] = \
                bigram_frequencies[w1].get(w2, 0) + 1

    return bigram_frequencies

def bigram_frequencies_analysis(bigram_frequencies, top_n=10):
    total_bigrams = 0
    unique_bigrams = 0
    flat_bigrams = []

    for w1, next_words in bigram_frequencies.items():
        for w2, count in next_words.items():
            total_bigrams += count
            unique_bigrams += 1
            flat_bigrams.append(((w1, w2), count))

    sorted_bigrams = sorted(
        flat_bigrams,
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Total bigrams: {total_bigrams}")
    print(f"Unique bigrams: {unique_bigrams}\n")

    print(f"Top {top_n} most frequent bigrams:")
    for (w1, w2), count in sorted_bigrams[:top_n]:
        print(f"({w1}, {w2}) -> {count}")


def bigram_generate_next_word(next_words_dict):
    words = list(next_words_dict.keys())
    weights = list(next_words_dict.values())
    return random.choices(words, weights=weights)[0]

def bigram_inference(bigram_frequencies, current_word="<s>"):
    result = [current_word]
    while current_word != "</s>":
        next_words = bigram_frequencies[current_word]
        current_word = bigram_generate_next_word(next_words)
        result.append(current_word)
    return result

def word_list_print(words):
    result = ''
    for word in words:
        if word in ["<s>", "</s>"]:
            continue

        if word in [".", ",", "!", "?", ":", ";"]:
            result = result.rstrip()
            result += word + " "
        else:
            result += word + " "
    result = result.strip()
    if result:
        result = result[0].upper() + result[1:]

    print(result)

In [130]:
bigram_frequencies = compute_bigram_frequencies(sentences)
#bigram_frequencies_analysis(bigram_frequencies)
sentence = bigram_inference(bigram_frequencies)
word_list_print(sentence)

Some one strong that i can be seen earlier chapters i should guess, daughter.


## Trigram

In [131]:
def compute_trigram_frequencies(sentences):
    trigram_frequencies = {}

    for sentence in sentences:
        for w1, w2, w3 in zip(sentence, sentence[1:], sentence[2:]):
            key = (w1, w2)

            if key not in trigram_frequencies:
                trigram_frequencies[key] = {}

            trigram_frequencies[key][w3] = \
                trigram_frequencies[key].get(w3, 0) + 1

    return trigram_frequencies

def trigram_frequencies_analysis(trigram_frequencies, top_n=10):
    total_trigrams = 0
    unique_trigrams = 0
    flat_trigrams = []

    for (w1, w2), next_words in trigram_frequencies.items():
        for w3, count in next_words.items():
            total_trigrams += count
            unique_trigrams += 1
            flat_trigrams.append(((w1, w2, w3), count))

    sorted_trigrams = sorted(
        flat_trigrams,
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Total trigrams: {total_trigrams}")
    print(f"Unique trigrams: {unique_trigrams}\n")

    print(f"Top {top_n} most frequent trigrams:")
    for (w1, w2, w3), count in sorted_trigrams[:top_n]:
        print(f"({w1}, {w2}, {w3}) -> {count}")

def trigram_generate_next_word(next_words_dict):
    words = list(next_words_dict.keys())
    weights = list(next_words_dict.values())
    return random.choices(words, weights=weights)[0]

def trigram_inference(trigram_frequencies, current_words=("<s>", "<s>")):
    result = list(current_words)

    while True:
        key = (current_words[0], current_words[1])

        if key not in trigram_frequencies:
            break

        next_words = trigram_frequencies[key]
        next_word = trigram_generate_next_word(next_words)

        result.append(next_word)

        if next_word == "</s>":
            break

        current_words = (current_words[1], next_word)

    return result

In [133]:
trigram_frequencies = compute_trigram_frequencies(trigram_sentences)
trigram_frequencies_analysis(trigram_frequencies)
for i in range(10):
    sentence = trigram_inference(trigram_frequencies)
    word_list_print(sentence)

Total trigrams: 710129
Unique trigrams: 422504

Top 10 most frequent trigrams:
(<s>, <s>, ') -> 6374
(<s>, <s>, the) -> 2139
(,, ', said) -> 1910
(<s>, <s>, but) -> 1735
(<s>, <s>, he) -> 1640
(<s>, <s>, i) -> 1358
(<s>, <s>, ") -> 1138
(<s>, <s>, and) -> 1134
(<s>, <s>, it) -> 1070
(<s>, <s>, they) -> 1017
The fire and the white horse.
Come inside at least of rings, does it not true of gimli, ' said strider reluctantly, 'you may join my mess for this scroll, ' said pippin, staring, while his talk i gathered at last to mithlond, to control it.
Forcing himself to be sure of that which is not our custom.
Not expecting even this story opens, so i jumped out in a spider's treacherous web!
In each farthing, for they had dared.
' he mocked.
Midyear's day, to pay attention to hobbiton at the far downs.
' asked frodo.
The night at the head of this horrible smell!
No onslaught more fierce than fire laid low their towers and the cries and howls getting closer and closer.
